[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/optical_rotation/optical_rotation.ipynb)

# Optical Rotation, Out and Back

This example computes the polarization of light travelling out and back through quartz and through a magnetized Faraday rotator, alongside the mirror image of each setup. Quartz and an axial magnetic field both rotate linear polarization on a single pass, yet a reflector at the far end separates the two mechanisms: quartz unwinds its outgoing turn on the return trip, whereas the magnetic field keeps turning in the same lab sense so that returning reflections arrive crossed with the entrance polarizer in an optical isolator.

In spacetime geometric algebra, the electric field is modelled as an extrinsic bivector and a polarizer reads it by pairing with the worldsheet plane of its axis, `field & (axis & time)`. Quartz turns the field in the intrinsic spacetime plane `(direction & time) * rotary_power`, where the chiral rotary power is an intrinsic scalar whose product with `direction & time` forms an extrinsic turn rate that reverses when `-direction` replaces `direction` on the return leg. A magnetic field is an extrinsic spatial bivector that turns the polarization via the magnetic flux `magnetic_field & cross_section` through the beam cross-section, which stays unchanged when the beam reverses, and reflecting every input via `mirror >> x` mirrors the full round-trip field at every depth.

In Jones calculus and Gibbs–Heaviside vector notation, a polarizing beam frame $(\hat{\mathbf{e}}_1, \hat{\mathbf{e}}_2, \hat{\mathbf{k}})$ flips handedness or reverses $\hat{\mathbf{e}}_1 \times \hat{\mathbf{e}}_2 = \hat{\mathbf{k}}$ upon reflection, requiring separate sign conventions for the specific rotation matrix $R(\pm \rho L)$ of a chiral crystal and the Faraday matrix $R(V (\mathbf{B}\cdot\hat{\mathbf{k}}) L)$ of an axial pseudovector field $\mathbf{B}$ so that the lab-frame round trips evaluate to $R(-\rho L)R(\rho L) = I$ and $R(V B_z L)R(V B_z L) = R(2 V B_z L)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.util.animation import save_figure
from examples.orientation.optical_rotation import render, scenarios

# Spacetime `x+y+z+t-`, with `+` extrinsic and `-` intrinsic.
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                       # Scalar+
Direction = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                        # Vector+
Field = ga.gatype.bivector()                                                      # Bivector+
TurnRate = ga.gatype.bivector()                                                   # Bivector+
CrossSection = ga.gatype.antibivector()                                           # Antibivector-
RotaryPower = ga.gatype.antipseudoscalar()                                        # Antipseudoscalar-

# Unit timelike direction of the medium rest frame, pairing to +1 with `mv.t`.
time = -mv.t.dual()                                                               # [] Direction-
# Unit electric field polarized along a spatial direction, dual to the spacetime plane swept out in time.
polarization_field = (Direction & time).dual()                                    # [] Field+ <- Direction-

## Round-trip rotation in chiral and magnetized media

An optically active crystal such as quartz has a handed crystal structure quantified by an intrinsic rotary power, which combines with the intrinsic travel-time sheet `direction & time` into an extrinsic spatial turn rate that reverses on the return pass. A Faraday rotator couples an extrinsic magnetic bivector to the transverse beam patch through `magnetic_field & cross_section`, producing the same turn rate on both passes so two 45-degree passes leave the returning wave orthogonal to the input polarizer, where Malus's law `(field & (axis & time)).squared()` gives zero transmission.

In [ ]:
path_length = scenarios.path_length
n_depths = scenarios.n_depths
depths = np.linspace(0, path_length, n_depths)
verdet_constant = scenarios.verdet_constant

propagation_direction = mv.z.dual()                                               # [] Direction-
polarizer_axis = mv.x.dual()                                                      # [] Direction-
cross_section = mv.xy.dual()                                                      # [] CrossSection-
incident_field = polarization_field(polarizer_axis)                               # [] Field+
rotary_power = mv(RotaryPower, [scenarios.quartz_rotary_power])                   # [] RotaryPower-
magnetic_field = mv.xy * scenarios.field_strength                                 # [] Field+

In [ ]:
def round_trip_fields(outgoing_rate: TurnRate, returning_rate: TurnRate, incident_field: Field,
                      depths: np.ndarray) -> Field:
    """Propagating `incident_field` over `depths` (`[n_depths]`) at `outgoing_rate` to a far-end reflector
    and back at `returning_rate` traces the outgoing and returning electric fields (`[2, n_depths] Field+`)."""
    outgoing_rotors = (outgoing_rate * (-depths / 2)).exp()                       # [n_depths] Rotor+
    outgoing = outgoing_rotors >> incident_field                                  # [n_depths] Field+
    returning_rotors = (returning_rate * (-depths / 2)).exp()                     # [n_depths] Rotor+
    returning = returning_rotors >> outgoing[-1]                                  # [n_depths] Field+
    return stack([outgoing, returning])                                           # [2, n_depths] Field+


def media_round_trips(direction: Direction, cross_section: CrossSection, rotary_power: RotaryPower,
                      magnetic_field: Field, verdet_constant: float, incident_field: Field,
                      depths: np.ndarray) -> Field:
    """Propagating `incident_field` out and back across `depths` (`[n_depths]`) through a chiral medium
    and a Faraday rotator yields both media's outgoing and returning fields (`[2, 2, n_depths] Field+`)."""
    # Multiplying the intrinsic travel-time plane by the intrinsic rotary power yields an extrinsic spatial turn rate.
    outgoing_sweep, returning_sweep = direction & time, -direction & time         # [] Antibivector-, Antibivector-
    chiral_legs = round_trip_fields(outgoing_sweep * rotary_power, returning_sweep * rotary_power,
                                    incident_field, depths)                       # [2, n_depths] Field+
    # Magnetic flux through the transverse cross-section drives the same spatial turn rate on both legs.
    faraday_rate = cross_section.dual_inverse() * (magnetic_field & cross_section) * verdet_constant  # [] TurnRate+
    faraday_legs = round_trip_fields(faraday_rate, faraday_rate, incident_field, depths)  # [2, n_depths] Field+
    return stack([chiral_legs, faraday_legs])                                     # [2, 2, n_depths] Field+


original_fields = media_round_trips(
    propagation_direction, cross_section, rotary_power, magnetic_field, verdet_constant, incident_field, depths
)                                                                                 # [2, 2, n_depths] Field+
transmitted_shares = (original_fields[:, 1, -1] & (polarizer_axis & time)).squared()  # [2] Scalar+
for medium_name, share in zip(("Quartz", "Magnetic field"), transmitted_shares):
    print(f"{medium_name}: {float(share.to_array()):.3f}")

## Mirror-reflected isolator setups

Reflecting the entire experiment across a plane containing the beam reflects the propagation direction, the cross-section, the magnetic field, the input polarization, and the crystal's rotary power alike. Right-handed quartz reflects into left-handed quartz because rotary power is an intrinsic scalar, whereas the magnetic bivector reflects as a spatial plane, so the reflected setups reproduce the exact mirror image of the outgoing and returning waves at every depth.

In [ ]:
mirror = mv.x                                                                     # [] Plane+
mirrored_fields = media_round_trips(
    mirror >> propagation_direction, mirror >> cross_section, mirror >> rotary_power,
    mirror >> magnetic_field, verdet_constant, mirror >> incident_field, depths
)                                                                                 # [2, 2, n_depths] Field+

fields = stack([original_fields, mirrored_fields], axis=1)                        # [2, 2, 2, n_depths] Field+
figure = render.isolator_figure(fields, depths)
save_figure(figure, "optical_rotation")
figure

Each panel unfolds the beam from left to right across the translucent far-end reflector, with quartz on the left, the Faraday rotator on the right, and the original setups above their mirror images across the vertical plane containing the beam. Orange rods mark the electric polarization on the outgoing pass, teal rods mark the polarization on the return pass, and dark dashed segments at both ends mark the entrance polarizer's axis.

In [ ]:
# checks
# Quartz restores the input field after a round trip, while the Faraday rotator returns light crossed with the polarizer.
chiral_exit, faraday_exit = original_fields[:, 1, -1]                             # [] Field+, Field+
mirror_mismatch = (mirrored_fields - (mirror >> original_fields)).scalar_norm_squared()  # [2, 2, n_depths] Scalar+
assert ((chiral_exit - incident_field).scalar_norm_squared().abs() < 1e-12).all()
assert ((faraday_exit & (polarizer_axis & time)).abs() < 1e-12).all()
assert (mirror_mismatch.abs() < 1e-12).all()